In [ ]:
import pyvisa
from  keithley2600 import Keithley2600
import time
import logging
import csv
import os
from datetime import datetime

    # # ======
    # # Logger
    # # ======
# init logger
format = "%(asctime)s: %(message)s"
log_file_path = 'example.log'
logging.basicConfig(format=format, level=logging.INFO,  
                        datefmt="%H:%M:%S", filename= log_file_path, filemode= 'w')

    # # ======
    # # Keithley
    # # ======
# init logger
keithley_instrument = Keithley2600('USB0::0x05E6::0x2636::4480001::INSTR', visa_library = 'C:/windows/System32/visa64.dll', timeout = 100000)
keithley_instrument.smua.source.output = keithley_instrument.smua.OUTPUT_OFF
keithley_instrument.smub.source.output = keithley_instrument.smub.OUTPUT_OFF

In [ ]:
#-- Reset SourceMeter instrument to default conditions.
keithley_instrument.reset()

#-- Clear the front panel display then prompt for input parameters if missing. 
keithley_instrument.display.clear() 

# -- Generate a single pulse with the following characteristics:
# -- * Bias (idle) level = 0 V
# -- * Pulse level = 0.2 V
# -- * Pulse width = 20 ms
# -- Configure the source function.
bias = 0
pulse_voltage = 0.2
ton = 0.2 # [s], pulse on time
toff = 0.8 # [s], pulse off time
n_pulses = 20 
# -- Update display with test info. 
keithley_instrument.display.settext("PulseV")  #-- Line 1 (20 characters max) 

#-- Configure source and measure settings (drain). 

keithley_instrument.smua.source.output = keithley_instrument.smua.OUTPUT_OFF 

if abs(pulse_voltage) > abs(bias):

    keithley_instrument.smua.source.rangev = pulse_voltage 

else: 

    keithley_instrument.smua.source.rangev = bias 

keithley_instrument.smua.source.func = keithley_instrument.smua.OUTPUT_DCVOLTS
#-- Set the voltage source range and the idle or bias source level and limit.
keithley_instrument.smua.source.levelv = pulse_voltage
keithley_instrument.smua.source.limiti = 0.1
# -- Use trigger timer 1 to control the period and trigger timer 2 to control the 
# -- pulse width. Alias the timers for convenience and clarity.
period_timer = keithley_instrument.trigger.timer[1]
pulsewidth_timer = keithley_instrument.trigger.timer[2]
# -- Configure the period timer to output 10 total trigger events.
period_timer.delay = 500e-3
# -- The effective count is 10 because the passthrough setting is true.
period_timer.count = 9
# -- Configure the timer to immediately output a trigger event when it is started.
keithley_instrument.period_timer.passthrough = True
# -- Start the timer when the SMU moves from the ARM layer to the TRIGGER layer.
period_timer.stimulus = keithley_instrument.smua.trigger.ARMED_EVENT_ID
# -- Configure the pulse width timer to output one trigger event for each period.
pulsewidth_timer.delay = 100e-3
pulsewidth_timer.count = 1
# -- Do not immediately output a trigger event when pulse width timer is started.
pulsewidth_timer.passthrough = False
# -- Start the pulse width timer with the period timer output trigger event.
pulsewidth_timer.stimulus = period_timer.EVENT_ID
# -- Configure the trigger model to execute a 10-point fixed-level voltage pulse 
# -- train. No measurements are made.
keithley_instrument.smua.trigger.source.listv({pulse_voltage})
keithley_instrument.smua.trigger.source.action = keithley_instrument.smua.ENABLE
keithley_instrument.smua.trigger.measure.action = keithley_instrument.smua.DISABLE
# -- Set the trigger source limit, which can be different than the bias limit.
# -- This is an important setting for pulsing in the extended operating area.
keithley_instrument.smua.trigger.source.limiti = 0.1
keithley_instrument.smua.measure.rangei = 0.1
# -- Trigger SMU source action with the period timer event.
keithley_instrument.smua.trigger.source.stimulus = period_timer.EVENT_ID
# -- Configure the endpulse action to achieve a pulse.
keithley_instrument.smua.trigger.endpulse.action = keithley_instrument.smua.SOURCE_IDLE
# -- Trigger the SMU end pulse action with a pulse width timer event.
keithley_instrument.smua.trigger.endpulse.stimulus = pulsewidth_timer.EVENT_ID
# -- Set the trigger model count to generate one 10-point pulse train.
keithley_instrument.smua.trigger.arm.count = 1
keithley_instrument.smua.trigger.count = 10
# -- Turn on the SMU output and initiate the trigger model to output the pulse train.
keithley_instrument.smua.source.output = keithley_instrument.smua.OUTPUT_ON
keithley_instrument.smua.trigger.initiate()
# -- Wait for the sweep to complete.
keithley_instrument.waitcomplete()
# -- Turn off SMU output.
# smua.source.output = smua.OUTPUT_OFF
